Let’s explore how to work with tools, using the Python REPL tool as an example. The Python REPL tool can run Python commands. These commands can either come from the user or the LLM can generate the commands. This tool is particularly useful for complex calculations. Instead of having the LLM generate the answer directly, using the LLM to generate code to calculate the answer is more efficient.

In [4]:
!pip install -q langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [5]:
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL

In [6]:
python_repl = PythonREPL()

In [7]:
python_repl.run("print(1+1)")

'2\n'

The `@tool` decorator is a convenient way to define tools, but you can also use the Tool class directly:

In [8]:
python_repl = PythonREPL()

python_calculator = Tool(

    name="Python Calculator",

    func=python_repl.run,

    description="Useful for when you need to perform calculations or execute Python code. Input should be valid Python code."
)

In [9]:
python_calculator.invoke("a = 3; b = 1; print(a+b)")

'4\n'

In [11]:
python_calculator.invoke(" print(squared of 5)")

"SyntaxError('invalid syntax. Perhaps you forgot a comma?', ('<string>', 1, 7, 'print(squared of 5)\\n', 1, 17))"

We can also create custom tools using the @tool decorator:

In [12]:
@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    # In a real application, this would call a weather API
    return f"The weather in {location} is currently sunny and 72°F."

# **Agents**

By themselves, language models can't take actions; they just output text. A big use case for LangChain is creating agents. Agents are systems that leverage a large language model (LLM) as a reasoning engine to identify appropriate actions and determine the required inputs for those actions. The results of those actions are to be fed back into the agent. The agent then makes a determination whether more actions are needed, or if the task is complete.

The modern approach to creating agents in LangChain uses the create_react_agent function and AgentExecutor:

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import Tool

In [18]:
# Create the ReAct agent prompt template
# The ReAct prompt needs to instruct the model to follow the thought-action-observation pattern
prompt_template = """You are an agent who has access to the following tools:

{tools}

The available tools are: {tool_names}

To use a tool, please use the following format:
```
Thought: I need to figure out what to do
Action: tool_name
Action Input: the input to the tool
```

After you use a tool, the observation will be provided to you:
```
Observation: result of the tool
```

Then you should continue with the thought-action-observation cycle until you have enough information to respond to the user's request directly.
When you have the final answer, respond in this format:
```
Thought: I know the answer
Final Answer: the final answer to the original query
```

Remember, when using the Python Calculator tool, the input must be valid Python code.

Begin!

Question: {input}
{agent_scratchpad}
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

In [21]:
@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    # In a real application, this would call a weather API
    return f"The weather in {location} is currently sunny and 72°F."

tools = [python_calculator, search_weather]

In [22]:
!pip install -U -q "langchain[huggingface]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 14.4 MB/s eta 0:00:00


In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "Qwen/Qwen3-Embedding-0.6B",
    model_provider="huggingface",
    temperature=0.7,
    max_tokens=1024,
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# **New Version Langchain**

In [27]:
!pip install -U -q langchain langchain-core langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.8 MB/s eta 0:00:00


In [4]:
!pip install -q --upgrade langchain langchain-core

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor


prompt_template = """You are an agent who has access to the following tools:

{tools}

The available tools are: {tool_names}

Question: {input}
{agent_scratchpad}
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    return f"The weather in {location} is currently sunny and 72°F."

@tool
def python_calculator(code: str):
    """Executes Python code and returns the result."""
    try:
        return str(eval(code))
    except Exception as e:
        return str(e)

tools = [search_weather, python_calculator]

agent = create_tool_calling_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


agent_executor.invoke({"input": "What is 5 + 3?"})

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)